# glassbox — architecture ablation

Six runs on Tiny Shakespeare, identical in every respect except one switch.
The baseline is the Phase 1 architecture; four runs each flip exactly one
thing; the last flips all four together.

Roughly **25 minutes total** on a T4. The output is a table of what each
switch is worth on its own, a checkpoint per variant, and a `results.json`
the visualizer reads.

Two deliberate choices about fairness:

- **Float32 throughout.** Mixed precision would be faster, but we are
  measuring differences of a few hundredths and fp16 rounding is noise on
  that scale.
- **Bias stays on everywhere, including the combined run.** The Phase 2
  model turned biases off as well, so its 1.4813 is not reproduced here —
  that is the point. Exactly four variables move, one at a time.


## 1 · Setup


In [2]:
import os, subprocess, json, time

ROOT = "/content/glassbox"
if os.path.isdir(ROOT):
    subprocess.run(["git", "-C", ROOT, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/udit-rawat/glassbox.git", ROOT], check=True)
os.chdir(ROOT)
subprocess.run(["pip", "install", "-q", "-e", ".", "--no-deps"], check=True)

from google.colab import drive
drive.mount("/content/drive")

ABLATION_DIR = "/content/drive/MyDrive/glassbox/ablation"
os.makedirs(ABLATION_DIR, exist_ok=True)

import torch
print("device      ", "cuda" if torch.cuda.is_available() else "cpu")
print("commit      ", subprocess.run(["git","log","--oneline","-1"],
                                      capture_output=True, text=True).stdout.strip())
print("out         ", ABLATION_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device       cuda
commit       71ddfef debug: export cell streams output instead of swallowing stderr
out          /content/drive/MyDrive/glassbox/ablation


## 2 · The six runs

Each variant differs from `baseline` by exactly one flag, except `all`.
Everything else — seed, iterations, learning rate, model size, dropout — is
held constant, which is the only reason the differences mean anything.


In [3]:
VARIANTS = [
    ("baseline", [],                                    "LayerNorm · GELU · learned · 6 kv"),
    ("rmsnorm",  ["--norm", "rmsnorm"],                 "RMSNorm only"),
    ("swiglu",   ["--activation", "swiglu"],            "SwiGLU only"),
    ("rope",     ["--pos-encoding", "rope"],            "RoPE only"),
    ("gqa",      ["--n-kv-heads", "2"],                 "2 kv heads only"),
    ("all",      ["--norm", "rmsnorm", "--activation", "swiglu",
                  "--pos-encoding", "rope", "--n-kv-heads", "2"], "all four"),
]

COMMON = ["--max-iters", "5000", "--lr", "1e-3", "--eval-interval", "500",
          "--seed", "1337", "--no-amp", "--sample-tokens", "300"]

for name, flags, label in VARIANTS:
    out = f"{ABLATION_DIR}/{name}"
    if os.path.exists(f"{out}/best.pt"):
        print(f"== {name:<9} already trained, skipping")
        continue

    print(f"\n{'=' * 62}\n== {name:<9} {label}\n{'=' * 62}", flush=True)
    cmd = ["python", "-u", "scripts/train_shakespeare.py",
           "--out-dir", out] + COMMON + flags
    t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        # Only the evaluations and the header, so six runs stay readable.
        if line.startswith(("iter", "best", "arch", "param", "precision")):
            print(line, end="")
    proc.wait()
    print(f"   {time.time() - t0:.0f}s", flush=True)



== baseline  LayerNorm · GELU · learned · 6 kv
arch        layernorm / gelu / learned / kv_heads 6
parameters  2,706,624
precision   fp32   effective batch 32 (32 x 1)   schedule constant
iter    500  train 2.0598  val 2.1217  lr 1.00e-03     23.3s
iter   1000  train 1.6839  val 1.8416  lr 1.00e-03     47.2s
iter   1500  train 1.5244  val 1.7122  lr 1.00e-03     73.8s
iter   2000  train 1.4452  val 1.6339  lr 1.00e-03    100.0s
iter   2500  train 1.3890  val 1.5864  lr 1.00e-03    124.9s
iter   3000  train 1.3472  val 1.5618  lr 1.00e-03    150.4s
iter   3500  train 1.3227  val 1.5505  lr 1.00e-03    176.1s
iter   4000  train 1.3072  val 1.5295  lr 1.00e-03    201.6s
iter   4500  train 1.2787  val 1.5039  lr 1.00e-03    227.0s
iter   5000  train 1.2727  val 1.5040  lr 1.00e-03    252.7s
best val loss 1.5039
   266s

== rmsnorm   RMSNorm only
arch        rmsnorm / gelu / learned / kv_heads 6
parameters  2,704,128
precision   fp32   effective batch 32 (32 x 1)   schedule constant
iter  

## 3 · Results

Read back out of the checkpoints rather than scraped from the logs, so the
table cannot disagree with the files.


In [ ]:
import torch
from glassbox.model import GPT

rows, results = [], {"corpus": "tinyshakespeare", "iters": 5000,
                     "precision": "fp32", "seed": 1337, "variants": []}

for name, flags, label in VARIANTS:
    path = f"{ABLATION_DIR}/{name}/best.pt"
    if not os.path.exists(path):
        continue
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = ck["model_config"]
    # Rebuilt and counted through parameters(), not by summing the state
    # dict. Weight tying puts the embedding under two keys —
    # token_embedding.weight and lm_head.weight — so summing the dict
    # counts it twice and inflates every variant by vocab_size * d_model.
    n = GPT(cfg).num_parameters()
    rows.append((name, label, ck["val_loss"], n))
    results["variants"].append({
        "name": name, "label": label, "flags": flags,
        "val_loss": ck["val_loss"], "params": n,
        "history": ck.get("history", []),
        "config": {"norm": cfg.norm, "activation": cfg.activation,
                    "pos_encoding": cfg.pos_encoding, "n_kv_heads": cfg.n_kv_heads,
                    "n_heads": cfg.n_heads, "n_layers": cfg.n_layers,
                    "d_model": cfg.d_model, "block_size": cfg.block_size,
                    "vocab_size": cfg.vocab_size, "bias": cfg.bias},
    })

base = dict((r[0], r[2]) for r in rows).get("baseline")
print(f"{'variant':<10} {'val loss':>9} {'vs base':>9} {'params':>12}   what changed")
print("-" * 74)
for name, label, loss, n in rows:
    delta = "" if name == "baseline" else f"{loss - base:+.4f}"
    print(f"{name:<10} {loss:>9.4f} {delta:>9} {n:>12,}   {label}")

with open(f"{ABLATION_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=1)
print(f"\nwrote {ABLATION_DIR}/results.json")


## 4 · Slim checkpoints for the visualizer

Optimizer state stripped, so each variant is small enough to download and
ship. These are what the generation panel loads.


In [5]:
for name, _, _ in VARIANTS:
    src = f"{ABLATION_DIR}/{name}/best.pt"
    if not os.path.exists(src):
        continue
    subprocess.run(["cp", src, "/content/tmp.pt"], check=True)
    r = subprocess.run(["python", "-u", "scripts/export_checkpoint.py",
                        "/content/tmp.pt", "--out", f"{ABLATION_DIR}/{name}_slim.pt"],
                       capture_output=True, text=True)
    print(f"{name:<10}", r.stdout.strip().splitlines()[-2] if r.stdout else r.stderr[-200:])


baseline   saved       21.7 MB  (67%)
rmsnorm    saved       21.7 MB  (67%)
swiglu     saved       21.7 MB  (67%)
rope       saved       21.5 MB  (67%)
gqa        saved       19.3 MB  (67%)
all        saved       19.1 MB  (67%)


## 5 · Bundle for download

One archive with every slim checkpoint, its tokenizer and `results.json` —
everything the visualizer needs, in a single file to move across.


In [ ]:
import shutil

bundle = "/content/ablation_bundle"
shutil.rmtree(bundle, ignore_errors=True)
os.makedirs(bundle, exist_ok=True)

shutil.copy(f"{ABLATION_DIR}/results.json", bundle)
for name, _, _ in VARIANTS:
    slim = f"{ABLATION_DIR}/{name}_slim.pt"
    if os.path.exists(slim):
        shutil.copy(slim, f"{bundle}/{name}.pt")
        shutil.copy(f"{ABLATION_DIR}/{name}/tokenizer.json", f"{bundle}/{name}_tokenizer.json")

archive = shutil.make_archive("/content/ablation", "zip", bundle)

# Copied into Drive rather than pushed at the browser. google.colab.files
# .download() is a browser API — it emits JavaScript for the page to execute.
# Through the VS Code extension there is no page, so it reports success and
# nothing arrives. Drive is the only way out of the runtime here.
dest = "/content/drive/MyDrive/glassbox/ablation_bundle.zip"
shutil.copy(archive, dest)

print(f"{dest}")
print(f"{os.path.getsize(dest) / 1e6:.1f} MB")
print()
print("Fetch it from drive.google.com > My Drive > glassbox > ablation_bundle.zip")
for f in sorted(os.listdir(bundle)):
    print(f"   {f}")
